# Functions

In [ ]:
import os
import pandas as pd
import numpy as np
import io
from scipy.io import arff
import tensorflow as tf
import math
import time

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, balanced_accuracy_score, precision_recall_curve, auc)
from imblearn.metrics import geometric_mean_score

from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm, neighbors
from lightgbm import LGBMClassifier
from sklearn.base import clone

import re
############################ Data feature names cleaning ############################
# This function cleans 'thoughts ?' -> 'thoughts?' and handles extra spaces
def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # 1. Remove leading/trailing whitespace
        c = col.strip()
        # 2. Replace multiple spaces with a single space
        c = re.sub(r'\s+', ' ', c)
        # 3. Remove space before punctuation (e.g., " ?" -> "?")
        c = re.sub(r'\s+([?.!,])', r'\1', c)
        new_columns.append(c)
    
    df.columns = new_columns
    return df

############################ Step 1: Identify Feature Types and Numericalize ############################
def identify_feature_types(df, t=15):
    metadata_list = []
    
    for col in df.columns:
        unique_vals = np.sort(df[col].unique())
        n_unique = len(unique_vals)
        dtype = df[col].dtype
        
        # --- Logic Gates for Categorization ---
        if n_unique <= t:
            # All low-cardinality features are treated as Categorical (Snapping required)
            category = 'Categ'
        
        else:
            if np.issubdtype(dtype, np.integer):
                # High-cardinality integers (Age, Study Hours)
                category = 'Num_disc'
            elif dtype == 'object':
                # High-cardinality strings (Names, IDs, Addresses)
                category = 'REMOVE'
            else:
                category = 'Num_cont' # Fallback for unknown numeric types
        
        metadata_list.append({
            'Feature': col,
            'Num Unique': n_unique,
            'Data Type': dtype,
            'Valid Values': unique_vals.tolist(),
            'Category': category
        })
    metadata = pd.DataFrame(metadata_list)
    Categ = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Categ']
    Num_disc = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_disc']
    Num_cont = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_cont']
    REMOVE = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'REMOVE']
        
    return metadata, Categ, Num_disc, Num_cont, REMOVE

def encode_to_numerical(df_org, REMOVE, Categ):
    # 1. Remove features in the REMOVE list
    df_cleaned = df_org.drop(REMOVE, axis=1)

    # 2. Initialize the translation dictionary (The "Rosetta Stone")
    # We will need this for Step 5 and Step 7
    mapping_dict = {}

    df_num = df_cleaned.copy()

    for col in Categ:
        # Get the valid values we stored in metadata
        # We sort them to ensure the mapping is consistent (e.g., 0 is always the same thing)
        unique_labels = sorted(df_num[col].unique())
        if col == Categ[-1]:
            # label = last column (always, Major = 0, Minor = 1)
            unique_labels = [df_num.iloc[:,-1].value_counts().index[0], df_num.iloc[:,-1].value_counts().index[1]]

        # Create the Forward (Text -> Num) and Reverse (Num -> Text) maps
        forward_map = {label: i for i, label in enumerate(unique_labels)}
        reverse_map = {i: label for i, label in enumerate(unique_labels)}

        # Save to our dictionary
        mapping_dict[col] = {
            'forward': forward_map,
            'reverse': reverse_map
        }

        # Apply the transformation
        df_num[col] = df_num[col].map(forward_map)

    # 3. Ensure all columns are numeric types (float or int) for SMOTE
    # This converts any remaining "object" types that might have slipped through
    df_num = df_num.apply(pd.to_numeric)
    
    return mapping_dict, df_num

In [ ]:
def get_results(y, predicted, pred_prob):
    # Initialize all variables to zero to prevent UnboundLocalError
    acc = pre = rec = spe = f1 = gmean = bacc = rauc = prauc = 0.0
    
    # Check for NaNs or empty predictions
    if np.isnan(predicted).any() or len(predicted) == 0:
        return acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc
    
    else:
        # 1. Efficient Confusion Matrix extraction
        # ravel() returns tn, fp, fn, tp in one line for binary classification
        try:
            tn, fp, fn, tp = metrics.confusion_matrix(y, predicted, labels=[0, 1]).ravel()
        except ValueError: 
            # Handles cases where only one class is present in y or predicted
            tn = tp = fp = fn = 0 

        # 2. Standard Metrics
        acc = np.round(accuracy_score(y, predicted), 4)
        pre = np.round(precision_score(y, predicted, zero_division=0), 4)
        rec = np.round(recall_score(y, predicted, zero_division=0), 4)
        
        # Specificity: TN / (TN + FP)
        spe = np.round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0.0
        
        f1 = np.round(f1_score(y, predicted, zero_division=0), 4)
        gmean = np.round(geometric_mean_score(y, predicted), 4)
        bacc = np.round(balanced_accuracy_score(y, predicted), 4)
        
        # 3. Probability-based Metrics (AUC)
        try:
            rauc = np.round(roc_auc_score(y, pred_prob), 4)
        except:
            rauc = 0.0
            
        try:
            precision, recall, _ = precision_recall_curve(y, pred_prob)
            prauc = np.round(auc(recall, precision), 4)
        except:
            prauc = 0.0
    
    return acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)   # 5-fold-cross validation

## Validation (70 => train:val = 56:14, 5CV) and Test(30)

In [ ]:
# 1. Define the Optimized Parameter Grids
configs = [
    # Logistic Regression (7) - Linear Classifier
    (LogisticRegression, {
        'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000], 
        'max_iter': [100000], 'random_state': [100]
    }),
    
    # Decision Tree - Pruned to focus on depth (36) - Tree-based Classifier
    (DecisionTreeClassifier, {
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 4, 6],
        'min_samples_leaf': [1, 2, 3], 
        'random_state': [100]
    }),
    
    # Random Forest - NEW (36) - Ensemble Classifier
    # Testing tree count vs depth vs leaf constraints
    (RandomForestClassifier, {
        'n_estimators': [50, 100, 150, 200], 
        'max_depth': [10, 20, None], 
        'min_samples_leaf': [1, 2, 4], 
        'random_state': [100],
        'n_jobs': [-1] # Faster training
    }),

    # SVM (24) - Distance-based Classifier
    (svm.SVC, {
        'C': [0.1, 1, 10], 'kernel': ['rbf', 'sigmoid'], 
        'gamma': ['scale', 'auto', 0.1, 1], 
        'probability': [True], 'random_state': [100]
    }),
               
    # KNN - Pruned to focus on local density (12) - Distance-based Classifier
    (neighbors.KNeighborsClassifier, {
        'n_neighbors': [3, 5, 7], 
        'p': [1, 2], 
        'metric': ['manhattan', 'minkowski']
    }),
               
    # LGBM (48) - Boosting-based Classifier
    (LGBMClassifier, {
        'boosting_type': ['gbdt', 'dart', 'goss'], 
        'max_depth': [10, 20], 
        'learning_rate': [0.01, 0.05], 
        'n_estimators': [50, 100, 150, 200], 
        'random_state': [100], 'verbose': [-1]
    })
]

# 2. Build Models and ModelNames in Parallel
Model = []
ModelName = []

config_map = {
    LogisticRegression: 'LR',
    DecisionTreeClassifier: 'DT',
    RandomForestClassifier: 'RF',
    svm.SVC: 'SVM',
    neighbors.KNeighborsClassifier: 'KNN',
    LGBMClassifier: 'LG'
}

for clf_class, params in configs:
    prefix = config_map[clf_class]
    grid = list(ParameterGrid(params))
    for i, p in enumerate(grid):
        Model.append(clf_class(**p))
        ModelName.append(f"{prefix}{i+1}")

print(f"Total Models: {len(Model)}") # Should be ~127 highly distinct models
print(ModelName)

In [ ]:
final_list = ['Wine', 'Depression', 'dmft-all', 'tourism-23457vs01', 
              'diu-ro10-cat', 'newthyroid1', 'page_blocks_1_3_vs_4']
path = './Dataset/'
Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]

# Experiments (ORG/SMO/BSM/ADA)

In [ ]:
for data in final_list:
    start = time.time()
    if data == 'Wine' or data == 'Depression':
        df = pd.read_csv(path + str(data) +'.csv')                                   # csv files
    else:
        with open(os.path.join(path+'Small/', str(data)+'.arff'), 'r') as f:         # arff files
            content = f.read().replace(', ', ',').replace(' ,', ',')
        rawdata, meta = arff.loadarff(io.StringIO(content))
        df = pd.DataFrame(rawdata)
        df = df.apply(lambda x: x.str.decode('utf8') if x.dtype == "object" else x)
    df = clean_column_names(df)                                                         # Clean Feature Names
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)      # Identifying Feature Types
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ)                           # Numericalizing the dataset
    print('+'*35, '{} Dataset'.format(data), '+'*35)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "1:{: .2f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
   
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]        # For validation (70%)
    y = df_val.iloc[:, -1]         # For validation (70%)
    X_test = df_test.iloc[:, :-1]  # For test (30%)
    y_test = df_test.iloc[:, -1]   # For test (30%)
    X_test = np.array(X_test).astype(float)
    y_test = np.array(y_test).astype(float)
    
    res_df = pd.DataFrame({'Dataset': [str(data) for a in range(27)]},
                   index = ['Acc_tr','Pre_tr','Rec_tr','Spe_tr','F1_tr','Gmean_tr','B_Acc_tr','R-AUC_tr','PR-AUC_tr',
                            'Acc_val','Pre_val','Rec_val','Spe_val','F1_val','Gmean_val','B_Acc_val','R-AUC_val','PR-AUC_val',
                            'Acc_t','Pre_t','Rec_t','Spe_t','F1_t','Gmean_t','B_Acc_t','R-AUC_t','PR-AUC_t'])
     
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    print("<min_strategy>:",min_strategy)  
    
    ##################### Original (Without Generation) #######################
    print("==========", "Original", "==========")
    for k in range(len(Model)):
        tr_acc = []
        tr_pre = []
        tr_rec = []
        tr_spe = []
        tr_f1 = []
        tr_gmean = []
        tr_bacc = []
        tr_rauc = []
        tr_prauc = []
        
        val_acc = []
        val_pre = []
        val_rec = []
        val_spe = []
        val_f1 = []
        val_gmean = []
        val_bacc = []
        val_rauc = []
        val_prauc = []
        
        # 5-fold-CV
        n_iter=0
        for train_index, val_index in skf.split(X, y):
            model = clone(Model[k])         # call the clean model
            n_iter += 1
            X_train = X.iloc[train_index]
            y_train= y.iloc[train_index]
            if k == 0 and n_iter == 1:
                print("TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))
            X_val = X.iloc[val_index]
            y_val= y.iloc[val_index]
            if k == 0 and n_iter == 1:
                print("VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
            # Array
            X_train = np.array(X_train).astype(float)
            y_train = np.array(y_train).astype(float)
            X_val = np.array(X_val).astype(float)
            y_val = np.array(y_val).astype(float)
            # Learning
            model.fit(X_train, y_train)
            train_result = model.predict(X_train)
            train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
            acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
            tr_acc.append(acc)
            tr_pre.append(pre)
            tr_rec.append(rec)
            tr_spe.append(spe)
            tr_f1.append(f1)
            tr_gmean.append(gmean)
            tr_bacc.append(bacc)
            tr_rauc.append(rauc)
            tr_prauc.append(prauc) 
            # Results 
            result = model.predict(X_val)
            result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
            acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
            val_acc.append(acc)
            val_pre.append(pre)
            val_rec.append(rec)
            val_spe.append(spe)
            val_f1.append(f1)
            val_gmean.append(gmean)
            val_bacc.append(bacc)
            val_rauc.append(rauc) 
            val_prauc.append(prauc) 
        
        # Test
        if k == 0:
            print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))  
        model = Model[k]
        model.fit(np.array(X).astype(float), np.array(y).astype(float))
        result = model.predict(X_test)
        result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)
        
        res_df['{}'.format(ModelName[k])] = [np.mean(tr_acc),np.mean(tr_pre), np.mean(tr_rec), np.mean(tr_spe),np.mean(tr_f1),
                                             np.mean(tr_gmean), np.mean(tr_bacc), np.mean(tr_rauc), np.mean(tr_prauc),
                                             np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                                             np.mean(val_gmean), np.mean(val_bacc), np.mean(val_rauc), np.mean(val_prauc),
                                             acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
    
    ##################### With Data Generation #######################
    methods = ['SMO','BSM','ADA']
    for meth in methods:
        for j in range(len(Strategy)):
            print("==========", "{}_{}".format(meth, Strategy[j]), "==========") 
            for k in range(len(Model)): 
                tr_acc = []
                tr_pre = []
                tr_rec = []
                tr_spe = []
                tr_f1 = []
                tr_gmean = []
                tr_bacc = []
                tr_rauc = []
                tr_prauc = []

                val_acc = []
                val_pre = []
                val_rec = []
                val_spe = []
                val_f1 = []
                val_gmean = []
                val_bacc = []
                val_rauc = []
                val_prauc = []

                if min_strategy > Strategy[j]:
                    res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]

                else:               
                    # 5-fold-CV
                    n_iter=0
                    for train_index, val_index in skf.split(X, y):
                        model = clone(Model[k])         # call the clean model
                        n_iter += 1
                        X_train = X.iloc[train_index]
                        y_train= y.iloc[train_index]
                        if k == 0 and n_iter == 1:
                            print("TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))     
                        # Loading Generated Data
                        try:
                            over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/'+str(meth)+'/'
                                                  +str(data)+'_'+str(meth)+'_'+str(Strategy[j])+'_'+str(n_iter)+'th.csv')
                        except:
                            res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]
                            continue
                        # over_df = over_df.replace('False', False)
                        # over_df = over_df.replace('FALSE', False)
                        # over_df = over_df.fillna(df.mean())
                        X_train = over_df.iloc[:, :-1]
                        y_train = over_df.iloc[:, -1]     
                        if k == 0 and n_iter == 1:
                            print("TRAIN_over(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))    
                        X_val = X.iloc[val_index]
                        y_val= y.iloc[val_index]
                        if k == 0 and n_iter == 1:
                            print("VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
                        # Array
                        X_train = np.array(X_train).astype(float)
                        y_train = np.array(y_train).astype(float)
                        X_val = np.array(X_val).astype(float)
                        y_val = np.array(y_val).astype(float)
                        # Learning
                        model.fit(X_train, y_train)
                        train_result = model.predict(X_train)
                        train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
                        tr_acc.append(acc)
                        tr_pre.append(pre)
                        tr_rec.append(rec)
                        tr_spe.append(spe)
                        tr_f1.append(f1)
                        tr_gmean.append(gmean)
                        tr_bacc.append(bacc)
                        tr_rauc.append(rauc) 
                        tr_prauc.append(prauc) 
                        # Results 
                        result = model.predict(X_val)
                        result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
                        val_acc.append(acc)
                        val_pre.append(pre)
                        val_rec.append(rec)
                        val_spe.append(spe)
                        val_f1.append(f1)
                        val_gmean.append(gmean)
                        val_bacc.append(bacc)
                        val_rauc.append(rauc)
                        val_prauc.append(prauc)

                    # Test
                    model = Model[k]
                    # Loading Resmapled Data
                    try:
                        over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/'+str(meth)+'/'
                                              +str(data)+'_'+str(meth)+'_'+str(Strategy[j])+'_full.csv')
                    except:
                        res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]
                        continue
                    X_over = over_df.iloc[:, :-1]
                    y_over = over_df.iloc[:, -1]
                    if k == 0:
                        print("TRAIN_over(0/1/total):", list(y_over).count(0), list(y_over).count(1), len(y_over))
                        print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))
                    model.fit(np.array(X_over).astype(float), np.array(y_over).astype(float))
                    result = model.predict(X_test)
                    result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
                    acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)

                    res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [
                        np.mean(tr_acc),np.mean(tr_pre),np.mean(tr_rec),np.mean(tr_spe),np.mean(tr_f1),
                        np.mean(tr_gmean),np.mean(tr_bacc),np.mean(tr_rauc),np.mean(tr_prauc),
                        np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                        np.mean(val_gmean),np.mean(val_bacc),np.mean(val_rauc),np.mean(val_prauc),
                        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
            
    end = time.time()
    print(end-start)
    res_df.to_csv('./Results/OSBA_result.csv', mode = 'a', float_format='%.4g')

In [ ]:
res_df

# Experiments (ROS/SEN/STM)

In [ ]:
for data in final_list:
    start = time.time()
    if data == 'Wine' or data == 'Depression':
        df = pd.read_csv(path + str(data) +'.csv')                                   # csv files
    else:
        with open(os.path.join(path+'Small/', str(data)+'.arff'), 'r') as f:         # arff files
            content = f.read().replace(', ', ',').replace(' ,', ',')
        rawdata, meta = arff.loadarff(io.StringIO(content))
        df = pd.DataFrame(rawdata)
        df = df.apply(lambda x: x.str.decode('utf8') if x.dtype == "object" else x)
    df = clean_column_names(df)                                                         # Clean Feature Names
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)      # Identifying Feature Types
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ)                           # Numericalizing the dataset
    print('+'*35, '{} Dataset'.format(data), '+'*35)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "1:{: .2f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
   
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]        # For validation (70%)
    y = df_val.iloc[:, -1]         # For validation (70%)
    X_test = df_test.iloc[:, :-1]  # For test (30%)
    y_test = df_test.iloc[:, -1]   # For test (30%)
    X_test = np.array(X_test).astype(float)
    y_test = np.array(y_test).astype(float)
    
    res_df = pd.DataFrame({'Dataset': [str(data) for a in range(27)]},
                   index = ['Acc_tr','Pre_tr','Rec_tr','Spe_tr','F1_tr','Gmean_tr','B_Acc_tr','R-AUC_tr','PR-AUC_tr',
                            'Acc_val','Pre_val','Rec_val','Spe_val','F1_val','Gmean_val','B_Acc_val','R-AUC_val','PR-AUC_val',
                            'Acc_t','Pre_t','Rec_t','Spe_t','F1_t','Gmean_t','B_Acc_t','R-AUC_t','PR-AUC_t'])
     
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    print("<min_strategy>:",min_strategy)  
    
    ##################### With Data Generation #######################
    methods = ['ROS','SEN','STM']
    for meth in methods:
        for j in range(len(Strategy)):
            print("==========", "{}_{}".format(meth, Strategy[j]), "==========") 
            for k in range(len(Model)): 
                tr_acc = []
                tr_pre = []
                tr_rec = []
                tr_spe = []
                tr_f1 = []
                tr_gmean = []
                tr_bacc = []
                tr_rauc = []
                tr_prauc = []

                val_acc = []
                val_pre = []
                val_rec = []
                val_spe = []
                val_f1 = []
                val_gmean = []
                val_bacc = []
                val_rauc = []
                val_prauc = []

                if min_strategy > Strategy[j]:
                    res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]

                else:               
                    # 5-fold-CV
                    n_iter=0
                    for train_index, val_index in skf.split(X, y):
                        model = clone(Model[k])         # call the clean model
                        n_iter += 1
                        X_train = X.iloc[train_index]
                        y_train= y.iloc[train_index]
                        if k == 0 and n_iter == 1:
                            print("TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))     
                        # Loading Generated Data
                        try:
                            over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/'+str(meth)+'/'
                                                  +str(data)+'_'+str(meth)+'_'+str(Strategy[j])+'_'+str(n_iter)+'th.csv')
                        except:
                            res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]
                            continue
                        # over_df = over_df.replace('False', False)
                        # over_df = over_df.replace('FALSE', False)
                        # over_df = over_df.fillna(df.mean())
                        X_train = over_df.iloc[:, :-1]
                        y_train = over_df.iloc[:, -1]     
                        if k == 0 and n_iter == 1:
                            print("TRAIN_over(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))    
                        X_val = X.iloc[val_index]
                        y_val= y.iloc[val_index]
                        if k == 0 and n_iter == 1:
                            print("VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
                        # Array
                        X_train = np.array(X_train).astype(float)
                        y_train = np.array(y_train).astype(float)
                        X_val = np.array(X_val).astype(float)
                        y_val = np.array(y_val).astype(float)
                        # Learning
                        try:
                            model.fit(X_train, y_train)
                        except:
                            tr_acc.append(0)
                            tr_pre.append(0)
                            tr_rec.append(0)
                            tr_spe.append(0)
                            tr_f1.append(0)
                            tr_gmean.append(0)
                            tr_bacc.append(0)
                            tr_rauc.append(0) 
                            tr_prauc.append(0)
                            val_acc.append(0)
                            val_pre.append(0)
                            val_rec.append(0)
                            val_spe.append(0)
                            val_f1.append(0)
                            val_gmean.append(0)
                            val_bacc.append(0)
                            val_rauc.append(0)
                            val_prauc.append(0)
                            continue
                        train_result = model.predict(X_train)
                        try:
                            train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
                        except:
                            train_result_prob = [0 for c in range(len(X_train))]
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
                        tr_acc.append(acc)
                        tr_pre.append(pre)
                        tr_rec.append(rec)
                        tr_spe.append(spe)
                        tr_f1.append(f1)
                        tr_gmean.append(gmean)
                        tr_bacc.append(bacc)
                        tr_rauc.append(rauc) 
                        tr_prauc.append(prauc) 
                        # Results 
                        result = model.predict(X_val)
                        try:
                            result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
                        except:
                            result_prob = [0 for c in range(len(X_val))]
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
                        val_acc.append(acc)
                        val_pre.append(pre)
                        val_rec.append(rec)
                        val_spe.append(spe)
                        val_f1.append(f1)
                        val_gmean.append(gmean)
                        val_bacc.append(bacc)
                        val_rauc.append(rauc)
                        val_prauc.append(prauc)

                    # Test
                    model = Model[k]
                    # Loading Resmapled Data
                    try:
                        over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/'+str(meth)+'/'
                                              +str(data)+'_'+str(meth)+'_'+str(Strategy[j])+'_full.csv')
                    except:
                        res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]
                        continue
                    X_over = over_df.iloc[:, :-1]
                    y_over = over_df.iloc[:, -1]
                    if k == 0:
                        print("TRAIN_over(0/1/total):", list(y_over).count(0), list(y_over).count(1), len(y_over))
                        print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))
                    try:
                        model.fit(np.array(X_over).astype(float), np.array(y_over).astype(float))
                    except:
                        res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [0 for c in range(27)]
                        continue
                    result = model.predict(X_test)
                    result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
                    acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)

                    res_df['{}_{}_{}'.format(meth, Strategy[j], ModelName[k])] = [
                        np.mean(tr_acc),np.mean(tr_pre),np.mean(tr_rec),np.mean(tr_spe),np.mean(tr_f1),
                        np.mean(tr_gmean),np.mean(tr_bacc),np.mean(tr_rauc),np.mean(tr_prauc),
                        np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                        np.mean(val_gmean),np.mean(val_bacc),np.mean(val_rauc),np.mean(val_prauc),
                        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
            
    end = time.time()
    print(end-start)
    res_df.to_csv('./Results/RSS_result.csv', mode = 'a', float_format='%.4g')

# Experiments (XEL {delta=1.5/2/2.5/3})

In [ ]:
for data in final_list:
    start = time.time()
    if data == 'Wine' or data == 'Depression':
        df = pd.read_csv(path + str(data) +'.csv')                                   # csv files
    else:
        with open(os.path.join(path+'Small/', str(data)+'.arff'), 'r') as f:         # arff files
            content = f.read().replace(', ', ',').replace(' ,', ',')
        rawdata, meta = arff.loadarff(io.StringIO(content))
        df = pd.DataFrame(rawdata)
        df = df.apply(lambda x: x.str.decode('utf8') if x.dtype == "object" else x)
    df = clean_column_names(df)                                                         # Clean Feature Names
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)      # Identifying Feature Types
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ)                           # Numericalizing the dataset
    print('+'*35, '{} Dataset'.format(data), '+'*35)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "1:{: .2f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]        # For validation (70%)
    y = df_val.iloc[:, -1]         # For validation (70%)
    X_test = df_test.iloc[:, :-1]  # For test (30%)
    y_test = df_test.iloc[:, -1]   # For test (30%)
    X_test = np.array(X_test).astype(float)
    y_test = np.array(y_test).astype(float)
    
    res_df = pd.DataFrame({'Dataset': [str(data) for a in range(27)]},
                   index = ['Acc_tr','Pre_tr','Rec_tr','Spe_tr','F1_tr','Gmean_tr','B_Acc_tr','R-AUC_tr','PR-AUC_tr',
                            'Acc_val','Pre_val','Rec_val','Spe_val','F1_val','Gmean_val','B_Acc_val','R-AUC_val','PR-AUC_val',
                            'Acc_t','Pre_t','Rec_t','Spe_t','F1_t','Gmean_t','B_Acc_t','R-AUC_t','PR-AUC_t'])
     
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    print("<min_strategy>:",min_strategy)  
    
    ##################### With Data Generation #######################
    delta_values = [1.5, 2, 2.5, 3]
    for delta in delta_values:
        for j in range(len(Strategy)):
            print("==========", "delta:{}_{}".format(delta, Strategy[j]), "==========") 
            for k in range(len(Model)): 
                tr_acc = []
                tr_pre = []
                tr_rec = []
                tr_spe = []
                tr_f1 = []
                tr_gmean = []
                tr_bacc = []
                tr_rauc = []
                tr_prauc = []

                val_acc = []
                val_pre = []
                val_rec = []
                val_spe = []
                val_f1 = []
                val_gmean = []
                val_bacc = []
                val_rauc = []
                val_prauc = []

                if min_strategy > Strategy[j]:
                    res_df['XEL(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [0 for c in range(27)]

                else:               
                    # 5-fold-CV
                    n_iter=0
                    for train_index, val_index in skf.split(X, y):
                        model = clone(Model[k])         # call the clean model
                        n_iter += 1
                        X_train = X.iloc[train_index]
                        y_train= y.iloc[train_index]
                        if k == 0 and n_iter == 1:
                            print("TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))     
                        # Loading Generated Data
                        over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/XEL_Trinity_Large/(delta='+str(delta)+')'
                                          +str(data)+'_XEL_'+str(Strategy[j])+'_'+str(n_iter)+'th.csv')
                        # over_df = over_df.replace('False', False)
                        # over_df = over_df.replace('FALSE', False)
                        # over_df = over_df.fillna(df.mean())
                        X_train = over_df.iloc[:, :-1]
                        y_train = over_df.iloc[:, -1]     
                        if k == 0 and n_iter == 1:
                            print("TRAIN_over(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))    
                        X_val = X.iloc[val_index]
                        y_val= y.iloc[val_index]
                        if k == 0 and n_iter == 1:
                            print("VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
                        # Array
                        X_train = np.array(X_train).astype(float)
                        y_train = np.array(y_train).astype(float)
                        X_val = np.array(X_val).astype(float)
                        y_val = np.array(y_val).astype(float)
                        # Learning
                        model.fit(X_train, y_train)
                        train_result = model.predict(X_train)
                        train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
                        tr_acc.append(acc)
                        tr_pre.append(pre)
                        tr_rec.append(rec)
                        tr_spe.append(spe)
                        tr_f1.append(f1)
                        tr_gmean.append(gmean)
                        tr_bacc.append(bacc)
                        tr_rauc.append(rauc) 
                        tr_prauc.append(prauc) 
                        # Results 
                        result = model.predict(X_val)
                        result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
                        val_acc.append(acc)
                        val_pre.append(pre)
                        val_rec.append(rec)
                        val_spe.append(spe)
                        val_f1.append(f1)
                        val_gmean.append(gmean)
                        val_bacc.append(bacc)
                        val_rauc.append(rauc)
                        val_prauc.append(prauc)

                    # Test
                    model = Model[k]
                    # Loading Resmapled Data
                    over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/XEL_Trinity_Large/(delta='+str(delta)+')'
                                          +str(data)+'_XEL_'+str(Strategy[j])+'_full.csv')
                    X_over = over_df.iloc[:, :-1]
                    y_over = over_df.iloc[:, -1]
                    if k == 0:
                        print("TRAIN_over(0/1/total):", list(y_over).count(0), list(y_over).count(1), len(y_over))
                        print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))
                    model.fit(np.array(X_over).astype(float), np.array(y_over).astype(float))
                    result = model.predict(X_test)
                    result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
                    acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)

                    res_df['XEL(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [
                        np.mean(tr_acc),np.mean(tr_pre),np.mean(tr_rec),np.mean(tr_spe),np.mean(tr_f1),
                        np.mean(tr_gmean),np.mean(tr_bacc),np.mean(tr_rauc),np.mean(tr_prauc),
                        np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                        np.mean(val_gmean),np.mean(val_bacc),np.mean(val_rauc),np.mean(val_prauc),
                        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
            
    end = time.time()
    print(end-start)
    res_df.to_csv('./Results/XEL(deltas)_result.csv', mode = 'a', float_format='%.4g')

# Ablation

# Experiments (X {delta=1.5/2/2.5/3})

In [ ]:
for data in final_list:
    start = time.time()
    if data == 'Wine' or data == 'Depression':
        df = pd.read_csv(path + str(data) +'.csv')                                   # csv files
    else:
        with open(os.path.join(path+'Small/', str(data)+'.arff'), 'r') as f:         # arff files
            content = f.read().replace(', ', ',').replace(' ,', ',')
        rawdata, meta = arff.loadarff(io.StringIO(content))
        df = pd.DataFrame(rawdata)
        df = df.apply(lambda x: x.str.decode('utf8') if x.dtype == "object" else x)
    df = clean_column_names(df)                                                         # Clean Feature Names
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)      # Identifying Feature Types
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ)                           # Numericalizing the dataset
    print('+'*35, '{} Dataset'.format(data), '+'*35)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "1:{: .2f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]        # For validation (70%)
    y = df_val.iloc[:, -1]         # For validation (70%)
    X_test = df_test.iloc[:, :-1]  # For test (30%)
    y_test = df_test.iloc[:, -1]   # For test (30%)
    X_test = np.array(X_test).astype(float)
    y_test = np.array(y_test).astype(float)
    
    res_df = pd.DataFrame({'Dataset': [str(data) for a in range(27)]},
                   index = ['Acc_tr','Pre_tr','Rec_tr','Spe_tr','F1_tr','Gmean_tr','B_Acc_tr','R-AUC_tr','PR-AUC_tr',
                            'Acc_val','Pre_val','Rec_val','Spe_val','F1_val','Gmean_val','B_Acc_val','R-AUC_val','PR-AUC_val',
                            'Acc_t','Pre_t','Rec_t','Spe_t','F1_t','Gmean_t','B_Acc_t','R-AUC_t','PR-AUC_t'])
     
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    print("<min_strategy>:",min_strategy)  
    
    ##################### With Data Generation #######################
    delta_values = [1.5, 2, 2.5, 3]
    for delta in delta_values:
        for j in range(len(Strategy)):
            print("==========", "delta:{}_{}".format(delta, Strategy[j]), "==========") 
            for k in range(len(Model)): 
                tr_acc = []
                tr_pre = []
                tr_rec = []
                tr_spe = []
                tr_f1 = []
                tr_gmean = []
                tr_bacc = []
                tr_rauc = []
                tr_prauc = []

                val_acc = []
                val_pre = []
                val_rec = []
                val_spe = []
                val_f1 = []
                val_gmean = []
                val_bacc = []
                val_rauc = []
                val_prauc = []

                if min_strategy > Strategy[j]:
                    res_df['X(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [0 for c in range(27)]

                else:               
                    # 5-fold-CV
                    n_iter=0
                    for train_index, val_index in skf.split(X, y):
                        model = clone(Model[k])         # call the clean model
                        n_iter += 1
                        X_train = X.iloc[train_index]
                        y_train= y.iloc[train_index]
                        if k == 0 and n_iter == 1:
                            print("TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))     
                        # Loading Generated Data
                        over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/X/(delta='+str(delta)+')'
                                          +str(data)+'_X_'+str(Strategy[j])+'_'+str(n_iter)+'th.csv')
                        # over_df = over_df.replace('False', False)
                        # over_df = over_df.replace('FALSE', False)
                        # over_df = over_df.fillna(df.mean())
                        X_train = over_df.iloc[:, :-1]
                        y_train = over_df.iloc[:, -1]     
                        if k == 0 and n_iter == 1:
                            print("TRAIN_over(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))    
                        X_val = X.iloc[val_index]
                        y_val= y.iloc[val_index]
                        if k == 0 and n_iter == 1:
                            print("VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
                        # Array
                        X_train = np.array(X_train).astype(float)
                        y_train = np.array(y_train).astype(float)
                        X_val = np.array(X_val).astype(float)
                        y_val = np.array(y_val).astype(float)
                        # Learning
                        model.fit(X_train, y_train)
                        train_result = model.predict(X_train)
                        train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
                        tr_acc.append(acc)
                        tr_pre.append(pre)
                        tr_rec.append(rec)
                        tr_spe.append(spe)
                        tr_f1.append(f1)
                        tr_gmean.append(gmean)
                        tr_bacc.append(bacc)
                        tr_rauc.append(rauc) 
                        tr_prauc.append(prauc) 
                        # Results 
                        result = model.predict(X_val)
                        result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
                        val_acc.append(acc)
                        val_pre.append(pre)
                        val_rec.append(rec)
                        val_spe.append(spe)
                        val_f1.append(f1)
                        val_gmean.append(gmean)
                        val_bacc.append(bacc)
                        val_rauc.append(rauc)
                        val_prauc.append(prauc)

                    # Test
                    model = Model[k]
                    # Loading Resmapled Data
                    over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/X/(delta='+str(delta)+')'
                                          +str(data)+'_X_'+str(Strategy[j])+'_full.csv')
                    X_over = over_df.iloc[:, :-1]
                    y_over = over_df.iloc[:, -1]
                    if k == 0:
                        print("TRAIN_over(0/1/total):", list(y_over).count(0), list(y_over).count(1), len(y_over))
                        print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))
                    model.fit(np.array(X_over).astype(float), np.array(y_over).astype(float))
                    result = model.predict(X_test)
                    result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
                    acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)

                    res_df['X(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [
                        np.mean(tr_acc),np.mean(tr_pre),np.mean(tr_rec),np.mean(tr_spe),np.mean(tr_f1),
                        np.mean(tr_gmean),np.mean(tr_bacc),np.mean(tr_rauc),np.mean(tr_prauc),
                        np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                        np.mean(val_gmean),np.mean(val_bacc),np.mean(val_rauc),np.mean(val_prauc),
                        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
            
    end = time.time()
    print(end-start)
    res_df.to_csv('./Results/X(deltas)_result.csv', mode = 'a', float_format='%.4g')

# Experiments (XE {delta=1.5/2/2.5/3})

In [ ]:
for data in final_list:
    start = time.time()
    if data == 'Wine' or data == 'Depression':
        df = pd.read_csv(path + str(data) +'.csv')                                   # csv files
    else:
        with open(os.path.join(path+'Small/', str(data)+'.arff'), 'r') as f:         # arff files
            content = f.read().replace(', ', ',').replace(' ,', ',')
        rawdata, meta = arff.loadarff(io.StringIO(content))
        df = pd.DataFrame(rawdata)
        df = df.apply(lambda x: x.str.decode('utf8') if x.dtype == "object" else x)
    df = clean_column_names(df)                                                         # Clean Feature Names
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)      # Identifying Feature Types
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ)                           # Numericalizing the dataset
    print('+'*35, '{} Dataset'.format(data), '+'*35)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "1:{: .2f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]        # For validation (70%)
    y = df_val.iloc[:, -1]         # For validation (70%)
    X_test = df_test.iloc[:, :-1]  # For test (30%)
    y_test = df_test.iloc[:, -1]   # For test (30%)
    X_test = np.array(X_test).astype(float)
    y_test = np.array(y_test).astype(float)
    
    res_df = pd.DataFrame({'Dataset': [str(data) for a in range(27)]},
                   index = ['Acc_tr','Pre_tr','Rec_tr','Spe_tr','F1_tr','Gmean_tr','B_Acc_tr','R-AUC_tr','PR-AUC_tr',
                            'Acc_val','Pre_val','Rec_val','Spe_val','F1_val','Gmean_val','B_Acc_val','R-AUC_val','PR-AUC_val',
                            'Acc_t','Pre_t','Rec_t','Spe_t','F1_t','Gmean_t','B_Acc_t','R-AUC_t','PR-AUC_t'])
     
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    print("<min_strategy>:",min_strategy)  
    
    ##################### With Data Generation #######################
    delta_values = [1.5, 2, 2.5, 3]
    for delta in delta_values:
        for j in range(len(Strategy)):
            print("==========", "delta:{}_{}".format(delta, Strategy[j]), "==========") 
            for k in range(len(Model)): 
                tr_acc = []
                tr_pre = []
                tr_rec = []
                tr_spe = []
                tr_f1 = []
                tr_gmean = []
                tr_bacc = []
                tr_rauc = []
                tr_prauc = []

                val_acc = []
                val_pre = []
                val_rec = []
                val_spe = []
                val_f1 = []
                val_gmean = []
                val_bacc = []
                val_rauc = []
                val_prauc = []

                if min_strategy > Strategy[j]:
                    res_df['XE(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [0 for c in range(27)]

                else:               
                    # 5-fold-CV
                    n_iter=0
                    for train_index, val_index in skf.split(X, y):
                        model = clone(Model[k])         # call the clean model
                        n_iter += 1
                        X_train = X.iloc[train_index]
                        y_train= y.iloc[train_index]
                        if k == 0 and n_iter == 1:
                            print("TRAIN(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))     
                        # Loading Generated Data
                        over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/XE/(delta='+str(delta)+')'
                                          +str(data)+'_XE_'+str(Strategy[j])+'_'+str(n_iter)+'th.csv')
                        # over_df = over_df.replace('False', False)
                        # over_df = over_df.replace('FALSE', False)
                        # over_df = over_df.fillna(df.mean())
                        X_train = over_df.iloc[:, :-1]
                        y_train = over_df.iloc[:, -1]     
                        if k == 0 and n_iter == 1:
                            print("TRAIN_over(0/1/total):", list(y_train).count(0), list(y_train).count(1), len(y_train))    
                        X_val = X.iloc[val_index]
                        y_val= y.iloc[val_index]
                        if k == 0 and n_iter == 1:
                            print("VALIDATION(0/1/total):", list(y_val).count(0), list(y_val).count(1), len(y_val))
                        # Array
                        X_train = np.array(X_train).astype(float)
                        y_train = np.array(y_train).astype(float)
                        X_val = np.array(X_val).astype(float)
                        y_val = np.array(y_val).astype(float)
                        # Learning
                        model.fit(X_train, y_train)
                        train_result = model.predict(X_train)
                        train_result_prob = model.predict_proba(X_train)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_train, train_result, train_result_prob)
                        tr_acc.append(acc)
                        tr_pre.append(pre)
                        tr_rec.append(rec)
                        tr_spe.append(spe)
                        tr_f1.append(f1)
                        tr_gmean.append(gmean)
                        tr_bacc.append(bacc)
                        tr_rauc.append(rauc) 
                        tr_prauc.append(prauc) 
                        # Results 
                        result = model.predict(X_val)
                        result_prob = model.predict_proba(X_val)[:, 1] # Get probability of class 1
                        acc, pre, rec, spe, f1, gmean, bacc, rauc, prauc = get_results(y_val, result, result_prob)
                        val_acc.append(acc)
                        val_pre.append(pre)
                        val_rec.append(rec)
                        val_spe.append(spe)
                        val_f1.append(f1)
                        val_gmean.append(gmean)
                        val_bacc.append(bacc)
                        val_rauc.append(rauc)
                        val_prauc.append(prauc)

                    # Test
                    model = Model[k]
                    # Loading Resmapled Data
                    over_df = pd.read_csv(r'./Generated Data/'+str(data)+'/XE/(delta='+str(delta)+')'
                                          +str(data)+'_XE_'+str(Strategy[j])+'_full.csv')
                    X_over = over_df.iloc[:, :-1]
                    y_over = over_df.iloc[:, -1]
                    if k == 0:
                        print("TRAIN_over(0/1/total):", list(y_over).count(0), list(y_over).count(1), len(y_over))
                        print("TEST(0/1/total):", list(y_test).count(0), list(y_test).count(1), len(y_test))
                    model.fit(np.array(X_over).astype(float), np.array(y_over).astype(float))
                    result = model.predict(X_test)
                    result_prob = model.predict_proba(X_test)[:, 1] # Get probability of class 1
                    acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t = get_results(y_test, result, result_prob)

                    res_df['XE(d:{})_{}_{}'.format(delta, Strategy[j], ModelName[k])] = [
                        np.mean(tr_acc),np.mean(tr_pre),np.mean(tr_rec),np.mean(tr_spe),np.mean(tr_f1),
                        np.mean(tr_gmean),np.mean(tr_bacc),np.mean(tr_rauc),np.mean(tr_prauc),
                        np.mean(val_acc),np.mean(val_pre),np.mean(val_rec),np.mean(val_spe),np.mean(val_f1),
                        np.mean(val_gmean),np.mean(val_bacc),np.mean(val_rauc),np.mean(val_prauc),
                        acc_t, pre_t, rec_t, spe_t, f1_t, gmean_t, bacc_t, rauc_t, prauc_t]
            
    end = time.time()
    print(end-start)
    res_df.to_csv('./Results/XE(deltas)_result.csv', mode = 'a', float_format='%.4g')